In [ ]:
! pip install transformers accelerate torch

First, will be trying with some sample text, next will try to sample some text and then run it.

In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


def generate_instruction(doc: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc)}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


if __name__ == "__main__":
    # The 'seed_docs' list is no longer needed as we'll use 'sampled'
    seed_docs = [
        "Vatten är nödvändigt för att kroppens organ ska fungera normalt och reglera temperaturen. Det hjälper också till att transportera näringsämnen och avlägsna slaggprodukter.",
        "En for-loop används för att upprepa en specifik block av kod ett bestämt antal gånger. Den är användbar när du vill gå igenom en lista av element eller köra samma instruktion sekventiellt tills ett villkor är uppfyllt.",
        "Hej! Tack för att du hör av dig. Vi beklagar verkligen att din nya kaffemaskin har blivit försenad under transporten. Vi har spårat ditt paket och det beräknas nu anlända till ditt ombud på torsdag; som kompensation har vi lagt till en rabattkod för ditt nästa köp i ett separat mejl.",
        "Det iskalla mörkret hade lagt sig över gruvstaden när det första larmet kom. Kapten Lindgren borstade bort snön från vindrutan och stirrade mot horisonten där blåljusen redan lyste upp natthimlen. Något var djupt begravt under permafrosten, och det var inte järnmalm."
    ]

    results = []
    for item in seed_docs: # Iterate over the 'sampled' list
        #doc_text = item['text'] # Extract the 'text' column from each item
        instruction = generate_instruction(item)
        results.append({"seed_doc": doc_text, "instruction": instruction}) # Store the text and generated instruction
        print(f"INSTRUKTION: {instruction}\n")
        print(f"Response: {doc_text}\n---")

    with open("instructions_sample.jsonl", "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

INSTRUKTION: Vad är vatten viktigt för i kroppen?
---
Response: Vatten är nödvändigt för att kroppens organ ska fungera normalt och reglera temperaturen. Det hjälper också till att transportera näringsämnen och avlägsna slaggprodukter.

INSTRUKTION: Vad är ett for-loop och vad används det till?
---
Response: En for-loop används för att upprepa en specifik block av kod ett bestämt antal gånger. Den är användbar när du vill gå igenom en lista av element eller köra samma instruktion sekventiellt tills ett villkor är uppfyllt.

INSTRUKTION: Hur går det med mitt paket?
---
Response: Hej! Tack för att du hör av dig. Vi beklagar verkligen att din nya kaffemaskin har blivit försenad under transporten. Vi har spårat ditt paket och det beräknas nu anlända till ditt ombud på torsdag; som kompensation har vi lagt till en rabattkod för ditt nästa köp i ett separat mejl.

INSTRUKTION: Beskriv en atmosfärisk och spännande inledning till en berättelse om en hemlig gruvupptäckt i en polarregion.
---
Re

Getting some sample data (Checking if we can scale it or not)

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
import random, json

random.seed(42)

ds = load_dataset("wikimedia/wikipedia", "20231101.sv", split="train", streaming=True)

MIN_WORDS, MAX_WORDS = 50, 100
N_SAMPLES = 25

sampled = []
seen = 0
for row in ds:
    text = row["text"].strip()
    n_words = len(text.split())
    if MIN_WORDS <= n_words <= MAX_WORDS:
        seen += 1
        if len(sampled) < N_SAMPLES:
            sampled.append({"id": row["id"], "title": row["title"], "text": text})
        else:
            j = random.randint(0, seen - 1)
            if j < N_SAMPLES:
                sampled[j] = {"id": row["id"], "title": row["title"], "text": text}
    if seen >= N_SAMPLES * 200:
        break

with open("seed_sample_pilot.jsonl", "w", encoding="utf-8") as f:
    for row in sampled:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Sampled {len(sampled)} seed documents -> seed_sample_pilot.jsonl")

In [ ]:
print(sampled)

[{'id': '9909', 'title': 'Gulfkriget (olika betydelser)', 'text': 'Gulfkriget kan syfta på:\n\n (Första) persiska gulfkriget – en väpnad konflikt mellan Irak och Iran som ägde rum från september 1980 till augusti 1988, se Iran–Irak-kriget\n Gulfkriget – ett fullskaligt krig mellan Irak och allierade trupper, ledda av USA 1990–1991, se Kuwaitkriget\n\nSe även\n Irakkriget – en militär kampanj i Irak, och invasionen av Irak 2003–2011\n Irakiska inbördeskriget – en konflikt i Irak 2011–'}, {'id': '70629', 'title': 'Zwinger', 'text': 'Zwinger i Dresden är ett barockkomplex som uppfördes 1710–1728 av arkitekten Matthäus Daniel Pöppelmann åt August den starke. Det var meningen att Zwinger skulle utgöra en del av ett planerat palatskomplex, men August dog innan det kunde realiseras.\n\nDen 13 och 14 februari 1945 förstördes stora delar av Dresden av allierat bombflyg. Zwinger skadades svårt, men återuppbyggandet började redan under hösten samma år. \n\nAllmänt åsyftar ordet "Zwinger" utrymmet

Medium scaled diverse data

In [ ]:
# for oscar data - optional

from huggingface_hub import login
login()  # paste your token when prompted

In [ ]:
from datasets import load_dataset
import random, json, requests, time

random.seed(42)

MIN_WORDS, MAX_WORDS = 50, 100
N_PER_SOURCE = 50

def reservoir_sample_from_stream(stream, source_name, n_per_source, min_words, max_words,
                                  overscan_factor=200, text_key="text", title_key="title", id_key="id"):
    sampled, seen = [], 0
    for row in stream:
        text = row[text_key].strip()
        n_words = len(text.split())
        if min_words <= n_words <= max_words:
            seen += 1
            item = {
                "id": row.get(id_key, seen),
                "title": row.get(title_key, ""),
                "text": text,
                "source": source_name,
            }
            if len(sampled) < n_per_source:
                sampled.append(item)
            else:
                j = random.randint(0, seen - 1)
                if j < n_per_source:
                    sampled[j] = item
        if seen >= n_per_source * overscan_factor:
            break
    print(f"{source_name}: sampled {len(sampled)} (scanned {seen} matching-length docs)")
    return sampled


# --- Source 1: Wikipedia (encyclopedic) ---
wiki_stream = load_dataset("wikimedia/wikipedia", "20231101.sv", split="train", streaming=True)
wiki_samples = reservoir_sample_from_stream(wiki_stream, "wikipedia", N_PER_SOURCE, MIN_WORDS, MAX_WORDS)

# --- Source 2: Wikisource (literary/archival register, same loader shape) ---
wikisource_stream = load_dataset("wikimedia/wikisource", "20231201.sv", split="train", streaming=True)
wikisource_samples = reservoir_sample_from_stream(wikisource_stream, "wikisource", N_PER_SOURCE, MIN_WORDS, MAX_WORDS)

# --- Source 3: mC4 Swedish (noisy web text, informal register) ---
mc4_stream = load_dataset("allenai/c4", "sv", split="train", streaming=True)
mc4_samples = reservoir_sample_from_stream(mc4_stream, "mc4_web", N_PER_SOURCE, MIN_WORDS, MAX_WORDS)

# --- Source 4: OSCAR Swedish (alternative web corpus, broader domain mix) ---
# Requires HF auth + accepting OSCAR's terms on the dataset page first.
# oscar_stream = load_dataset("oscar-corpus/OSCAR-2301", "sv", split="train", streaming=True, trust_remote_code=True)
# oscar_samples = reservoir_sample_from_stream(oscar_stream, "oscar_web", N_PER_SOURCE, MIN_WORDS, MAX_WORDS)

# --- Source 5: Riksdagen (parliamentary/formal-spoken register) ---
# Riksdagen's open data API doesn't stream like HF datasets, so pull docs directly.
def fetch_riksdagen_docs(n_docs, min_words, max_words, doktyp="mot", max_pages=30):
    """doktyp='mot' = motions; try 'prot' for debate protocols for a more spoken register."""
    samples, seen = [], 0
    base_url = "https://data.riksdagen.se/dokumentlista/"
    for page in range(1, max_pages + 1):
        params = {"doktyp": doktyp, "sort": "datum", "sortorder": "desc",
                   "utformat": "json", "p": page}
        resp = requests.get(base_url, params=params, timeout=30)
        resp.raise_for_status()
        docs = resp.json().get("dokumentlista", {}).get("dokument", [])
        if not docs:
            break
        for d in docs:
            text_url = d.get("dokument_url_text")
            if not text_url:
                continue
            try:
                text_resp = requests.get(text_url, timeout=30)
                text = text_resp.text.strip()
            except requests.RequestException:
                continue
            n_words = len(text.split())
            if min_words <= n_words <= max_words:
                seen += 1
                samples.append({"id": d.get("id"), "title": d.get("titel", ""),
                                 "text": text, "source": "riksdagen"})
                if len(samples) >= n_docs:
                    return samples
            time.sleep(0.2)  # be polite to the API
    print(f"riksdagen: sampled {len(samples)} (scanned {seen})")
    return samples

riksdagen_samples = fetch_riksdagen_docs(N_PER_SOURCE, MIN_WORDS, MAX_WORDS)

# --- Combine, shuffle, dedup ---
all_samples = wiki_samples + wikisource_samples + mc4_samples + riksdagen_samples

seen_texts = set()
deduped = []
for item in all_samples:
    key = item["text"][:200]  # cheap near-dedup on text prefix
    if key not in seen_texts:
        seen_texts.add(key)
        deduped.append(item)

random.shuffle(deduped)
print(f"Total after dedup: {len(deduped)} (from {len(all_samples)} raw)")

with open("seed_sample_combined.jsonl", "w", encoding="utf-8") as f:
    for row in deduped:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

sampled = deduped  # feed straight into your generation script below


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

wikipedia: sampled 50 (scanned 10000 matching-length docs)


README.md:   0%|          | 0.00/33.2k [00:00<?, ?B/s]

wikisource: sampled 50 (scanned 730 matching-length docs)


README.md:   0%|          | 0.00/41.1k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

mc4_web: sampled 50 (scanned 10000 matching-length docs)
riksdagen: sampled 0 (scanned 0)
Total after dedup: 150 (from 150 raw)


Normal Generation

In [ ]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


def generate_instruction(doc: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc)}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


if __name__ == "__main__":
    results = []
    for item in sampled:
      doc = item['text']
      instruction = generate_instruction(doc)
      results.append({"seed_doc": doc, "instruction": instruction})
      #print(f"INSTRUKTION: {instruction}\n")
      #print(f"Response: {doc}\n---")

    with open("instructions_sample.jsonl", "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

Generation for medium scaled data:- Owen 3.5-4B

In [ ]:
import json, os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from tqdm import tqdm

MODEL_ID = "Qwen/Qwen3.5-4B"
OUTPUT_PATH = "instructions_sample_Qwen.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


def generate_instruction(doc: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc)}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
    )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Resume support: skip ids already processed if the run was interrupted
already_done = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            already_done.add(json.loads(line)["id"])
    print(f"Resuming: {len(already_done)} already processed")

if __name__ == "__main__":
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:  # append, not overwrite
        for item in tqdm(sampled, desc="Generating instructions"):
            if item["id"] in already_done:
                continue
            doc = item["text"]
            try:
                instruction = generate_instruction(doc)
            except Exception as e:
                print(f"Failed on id={item['id']}: {e}")
                continue

            result = {
                "id": item["id"],
                "source": item.get("source", "unknown"),
                "seed_doc": doc,
                "instruction": instruction,


            }
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()  # write immediately so a crash doesn't lose the batch

Generation with qwen 2.5-3B Instruct

In [ ]:
import json
import os
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)


# ============================================================
# Configuration
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

INPUT_PATH = "/content/seed_sample_combined.jsonl"
OUTPUT_PATH = "/content/instructions_sample_Qwen2.5_3B.jsonl"


# ============================================================
# Check GPU
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required. "
        "In Colab, go to Runtime -> Change runtime type -> GPU."
    )

device = "cuda"

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


# ============================================================
# Load tokenizer
# ============================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

print("Tokenizer loaded.")


# ============================================================
# Load model
# ============================================================

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model loaded successfully.")


# ============================================================
# Prompt
# ============================================================

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


# ============================================================
# Generate instruction
# ============================================================

def generate_instruction(
    doc: str,
    max_new_tokens: int = 200
) -> str:

    messages = [
        {
            "role": "user",
            "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc),
        }
    ]

    # Qwen2.5 supports the standard Transformers chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    ).to(model.device)

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=max_new_tokens,

            # Sampling parameters
            do_sample=True,
            temperature=0.7,
            top_p=0.9,

            # Avoid unnecessary repetition
            repetition_penalty=1.05,

            eos_token_id=tokenizer.eos_token_id,
        )

    # Remove input tokens from generated output
    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[-1]:
    ]

    generated = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated


# ============================================================
# Resume support
# ============================================================

already_done = set()

if os.path.exists(OUTPUT_PATH):

    with open(
        OUTPUT_PATH,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            try:
                result = json.loads(line)

                if "id" in result:
                    already_done.add(result["id"])

            except json.JSONDecodeError:
                pass

    print(
        f"Resuming: {len(already_done)} already processed"
    )


# ============================================================
# Count total input items
# ============================================================

with open(
    INPUT_PATH,
    "r",
    encoding="utf-8"
) as f:

    total_items = sum(
        1
        for line in f
        if line.strip()
    )

print(
    f"Total input items: {total_items}"
)


# ============================================================
# Process input JSONL
# ============================================================

if __name__ == "__main__":

    with open(
        INPUT_PATH,
        "r",
        encoding="utf-8"
    ) as infile, \
    open(
        OUTPUT_PATH,
        "a",
        encoding="utf-8"
    ) as outfile:

        for line in tqdm(
            infile,
            total=total_items,
            desc="Generating instructions"
        ):

            if not line.strip():
                continue

            item = json.loads(line)

            item_id = item["id"]

            # Skip already processed items
            if item_id in already_done:
                continue

            doc = item["text"]

            try:

                instruction = generate_instruction(
                    doc,
                    max_new_tokens=200
                )

            except Exception as e:

                print(
                    f"\nFailed on id={item_id}: {e}"
                )

                continue

            result = {
                "id": item_id,
                "source": item.get(
                    "source",
                    "unknown"
                ),
                "seed_doc": doc,
                "instruction": instruction,
            }

            outfile.write(
                json.dumps(
                    result,
                    ensure_ascii=False
                ) + "\n"
            )

            # Save immediately so interrupted runs can resume
            outfile.flush()

            already_done.add(item_id)


print("Done!")
print(
    f"Output saved to: {OUTPUT_PATH}"
)

Generation with glm 5.3-flash

In [ ]:
!pip install -q "transformers==4.36.2" "tokenizers==0.15.2"

In [ ]:
import json
import os
from tqdm import tqdm
import torch
from transformers import AutoConfig, AutoModel, AutoTokenizer

MODEL_ID = "THUDM/chatglm2-6b"
INPUT_PATH = "/content/seed_sample_combined.jsonl"
OUTPUT_PATH = "instructions_sample_GLM.jsonl"

# Load config first to handle potential missing 'max_length'
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)

if not hasattr(config, "max_length"):
    if hasattr(config, "seq_length"):
        config.max_length = config.seq_length
        print(f"Set config.max_length from config.seq_length: {config.max_length}")
    else:
        config.max_length = 2048  # Fallback
        print(f"Set config.max_length to default: {config.max_length}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Determine device to load the model onto
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on device: {device}")

# Use AutoModel instead of AutoModelForCausalLM for GLM architecture compatibility
model = AutoModel.from_pretrained(
    MODEL_ID,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).to(device)

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


def generate_instruction(doc: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc)}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )
    generated = output_ids[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Resume support: skip ids already processed if the run was interrupted
already_done = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                already_done.add(json.loads(line)["id"])
    print(f"Resuming: {len(already_done)} already processed")

if __name__ == "__main__":
    # Count total lines in input file for tqdm progress bar
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        total_items = sum(1 for line in f if line.strip())

    # Read input line by line and append output
    with open(INPUT_PATH, "r", encoding="utf-8") as infile, \
         open(OUTPUT_PATH, "a", encoding="utf-8") as outfile:

        for line in tqdm(infile, total=total_items, desc="Generating instructions"):
            if not line.strip():
                continue

            item = json.loads(line)

            if item["id"] in already_done:
                continue

            doc = item["text"]
            try:
                instruction = generate_instruction(doc)
            except Exception as e:
                print(f"Failed on id={item['id']}: {e}")
                continue

            result = {
                "id": item["id"],
                "source": item.get("source", "unknown"),
                "seed_doc": doc,
                "instruction": instruction,
            }
            outfile.write(json.dumps(result, ensure_ascii=False) + "\n")
            outfile.flush()

Google Gemma 4

In [ ]:
from huggingface_hub import notebook_login
!pip install -U huggingface_hub

notebook_login()

In [ ]:
import json
import os
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/gemma-4-E4B-it"
INPUT_PATH = "/content/seed_sample_combined.jsonl"
OUTPUT_PATH = "instructions_sample_gemma.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,  # Use torch.float16 if bfloat16 causes issues on older Colab GPUs
    device_map="auto",
)

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


def generate_instruction(doc: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc)}]

    # Standard Chat Template for Gemma models
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

    generated = output_ids[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Resume support: skip ids already processed if the run was interrupted
already_done = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                already_done.add(json.loads(line)["id"])
    print(f"Resuming: {len(already_done)} already processed")

if __name__ == "__main__":
    # Count total valid lines in input file for tqdm progress bar
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        total_items = sum(1 for line in f if line.strip())

    # Stream read from seed_sample_combined.jsonl and append outputs
    with open(INPUT_PATH, "r", encoding="utf-8") as infile, \
         open(OUTPUT_PATH, "a", encoding="utf-8") as outfile:

        for line in tqdm(infile, total=total_items, desc="Generating instructions"):
            if not line.strip():
                continue

            item = json.loads(line)

            if item["id"] in already_done:
                continue

            doc = item["text"]
            try:
                instruction = generate_instruction(doc)
            except Exception as e:
                print(f"Failed on id={item['id']}: {e}")
                continue

            result = {
                "id": item["id"],
                "source": item.get("source", "unknown"),
                "seed_doc": doc,
                "instruction": instruction,
            }
            outfile.write(json.dumps(result, ensure_ascii=False) + "\n")
            outfile.flush()

Generation with qwen 3.8 27T model - 4 bit quantized (Not being used currently)

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import json, os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from tqdm import tqdm

!pip install -U bitsandbytes>=0.46.1

# Updated to the Qwen 3.8 27B model
MODEL_ID = "Qwen/Qwen3.8-27B"
OUTPUT_PATH = "instructions_sample.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Configure 4-bit quantization parameters to save VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # Normalized Float 4
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bfloat16 for stability
    bnb_4bit_use_double_quant=True      # Double quantization for maximum memory savings
)

# Load the 27B model using the 4-bit config
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

REVERSE_INSTRUCTION_PROMPT = """Nedan visas en text på svenska.

Text:
{doc}

Vilken typ av instruktion eller fråga skulle kunna ha gett denna text som svar? \
Skriv bara själva instruktionen, inget annat. Instruktionen ska vara naturlig svenska, \
inte en översättning från engelska.

Instruktion:"""


def generate_instruction(doc: str, max_new_tokens: int = 200) -> str:
    messages = [{"role": "user", "content": REVERSE_INSTRUCTION_PROMPT.format(doc=doc)}]
    # Qwen 3.8 uses enable_thinking=False to bypass reasoning tokens and save context window
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
    )
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


# Resume support: skip ids already processed if the run was interrupted
already_done = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            already_done.add(json.loads(line)["id"])
    print(f"Resuming: {len(already_done)} already processed")

if __name__ == "__main__":
    # Ensure your 'sampled' dataset variable is defined before executing
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:  # append, not overwrite
        for item in tqdm(sampled, desc="Generating instructions"):
            if item["id"] in already_done:
                continue
            doc = item["text"]
            try:
                instruction = generate_instruction(doc)
            except Exception as e:
                print(f"Failed on id={item['id']}: {e}")
                continue

            result = {
                "id": item["id"],
                "source": item.get("source", "unknown"),
                "seed_doc": doc,
                "instruction": instruction,
            }
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()  # write immediately so a crash doesn't lose the batch

##Evaluation

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import json
import os
import torch
from pydantic import BaseModel, Field
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Selected Judge Model from EuroEval top rankers
#JUDGE_MODEL_ID = "mistralai/Mistral-Small-24B-Instruct-2501"
JUDGE_MODEL_ID = "mistralai/Ministral-8B-Instruct"
INPUT_PATH = "instructions_sample.jsonl"
OUTPUT_PATH = "evaluation_results.jsonl"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

JUDGE_PROMPT_TEMPLATE = """Du är en expertutvärderare för svensk språkdata och instruktionsfinstämning.
Din uppgift är att bedöma kvaliteten på en AI-genererad instruktion baserad på en givna källtext (seed document).

---
[KÄLLTEXT / SEED DOC]:
{seed_doc}

[GENERERAD INSTRUKTION]:
{instruction}
---

Utvärdera följande två kriterier på en skala från 1 till 5:

1. **relevance_score** (1-5): Hur väl stämmer instruktionen överens med källtexten? Skulle källtexten utgöra ett naturligt och korrekt svar på instruktionen?
2. **idiomaticity_score** (1-5): Hur naturlig, korrekta och idiomatisk är den svenska språknivån i instruktionen? (Inget direktöversatt eller konstlat språk).

Svara EXAKT i följande JSON-format och inget annat:
{{
  "relevance_score": <heltal 1-5>,
  "idiomaticity_score": <heltal 1-5>,
  "reasoning": "<kort motivering på svenska>"
}}"""


def judge_pair(seed_doc: str, instruction: str) -> dict:
    prompt = JUDGE_PROMPT_TEMPLATE.format(seed_doc=seed_doc, instruction=instruction)
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,  # Greedy decoding for consistent LLM-as-a-judge evaluation
            temperature=0.0,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1] :]
    response_text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Parse judge output
    try:
        # Extract JSON blob if wrap in markdown code fence
        if "```json" in response_text:
            response_text = response_text.split("```json")[1].split("```")[0].strip()
        elif "```" in response_text:
            response_text = response_text.split("```")[1].split("```")[0].strip()

        parsed = json.loads(response_text)
        return parsed
    except Exception as e:
        return {
            "relevance_score": None,
            "idiomaticity_score": None,
            "reasoning": f"Failed to parse model response: {response_text}",
        }


def main():
    # Load processed records
    records = []
    if not os.path.exists(INPUT_PATH):
        print(f"Error: {INPUT_PATH} does not exist.")
        return

    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line.strip()))

    # Support resume logic
    already_evaluated = set()
    if os.path.exists(OUTPUT_PATH):
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                already_evaluated.add(json.loads(line)["id"])
        print(f"Resuming: {len(already_evaluated)} samples already evaluated.")

    # Evaluate each generated instruction pair
    with open(OUTPUT_PATH, "a", encoding="utf-8") as f:
        for item in tqdm(records, desc="Evaluating with LLM Judge"):
            item_id = item["id"]
            if item_id in already_evaluated:
                continue

            evaluation = judge_pair(
                seed_doc=item["seed_doc"], instruction=item["instruction"]
            )

            result = {
                "id": item_id,
                "seed_doc": item["seed_doc"],
                "instruction": item["instruction"],
                "judge_model": JUDGE_MODEL_ID,
                "eval_relevance": evaluation.get("relevance_score"),
                "eval_idiomaticity": evaluation.get("idiomaticity_score"),
                "eval_reasoning": evaluation.get("reasoning"),
            }

            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()


if __name__ == "__main__":
    main()

# Deduplicate and filter generated instructions
Remove near-duplicate instructions (embedding similarity or MinHash) and drop degenerate ones — too short, boilerplate ('Vad handlar texten om?' repeated everywhere), [not so sure how will it work out] or anything with leaked English/translation artifacts.